# Evidencia de ingesta de derivados

> Cuaderno de validación reproducible del pipeline ETL de derivados (Binance Futures) sobre SQL y artefactos `parquet` en disco.

## Contexto y motivación

Sobre la espina dorsal de velas OHLCV el sistema apila una segunda capa de mercado: los **derivados** de Binance Futures. Tres señales convencionales - *funding rate*, *open interest* y *long/short ratio* - capturan el **posicionamiento** y el **coste de financiación** del lado perpetuo, que es donde se concentra la mayor parte del flujo especulativo cripto y donde a menudo se anticipan giros que el mercado spot solo confirma *a posteriori*.

Este cuaderno materializa la siguiente evidencia como hace su homólogo `02_evidencia_ingesta_ohlcv.ipynb` (`notebooks/01_ingesta/ohlcv/`), pero adaptada a las tres `fact tables` de derivados:

- una **ejecución end-to-end** del validador (`scripts/validate_derivatives_pipeline.py`) con su matriz de aceptación,
- una **batería de consultas SQL** sobre `derivatives_data.*` que cuantifica cobertura, auditoría, duplicados e integridad **por tabla y símbolo**,
- un **inventario en disco** de los `parquet` raw bajo `data/01_raw/derivatives/` (uno por par símbolo×métrica),
- y la **exportación** de todo a `reports/validation/derivatives/` en formato JSON/CSV con `stamp` UTC, para citar números trazables desde la memoria.

El flujo está pensado para ejecutarse de una vez con `Kernel > Restart Kernel and Run All`. Cada sección añade una pieza de evidencia y termina con una breve interpretación.

## 1. Configuración del entorno

Se localiza la raíz del proyecto buscando hacia arriba desde el `cwd` actual hasta encontrar `scripts/validate_derivatives_pipeline.py`. Esto permite ejecutar el cuaderno tanto desde `notebooks/01_ingesta/derivados/` como desde la raíz del repositorio o desde el contenedor Docker (donde la raíz es `/app`).

También se capturan los **metadatos de ejecución** (timestamp UTC, versión de Python y plataforma). Sin ellos, mañana no sabríamos en qué entorno se generaron los CSV/JSON exportados al final.

In [1]:
from __future__ import annotations

import contextlib
import importlib
import io
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display
from sqlalchemy import text

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "scripts" / "validate_derivatives_pipeline.py").exists():
        ROOT = candidate
        break

assert ROOT is not None, "No se encontró la raíz del proyecto (falta scripts/validate_derivatives_pipeline.py)."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.validate_derivatives_pipeline as val
from src.utils.database import create_db_engine

val = importlib.reload(val)

SYMBOLS_OVERRIDE = None
DERIVATIVES_SCHEMA = "derivatives_data"
DERIVATIVES_FACT_TABLES = [
    "fact_funding_rate",
    "fact_open_interest_4h",
    "fact_long_short_ratio_4h",
]
DERIVATIVES_METRICS = ["funding_rate", "open_interest_4h", "long_short_ratio_4h"]
DERIVATIVES_RAW_DIR = ROOT / "data" / "01_raw" / "derivatives"
DERIVATIVES_OUTPUT_DIR = ROOT / "reports" / "validation" / "derivatives"
RUN_TIMESTAMP = datetime.now(timezone.utc)
RUN_STAMP = RUN_TIMESTAMP.strftime("%Y%m%dT%H%M%SZ")
DERIVATIVES_FREEZE_COVERAGE_DATE = val.DERIVATIVES_FREEZE_COVERAGE_DATE
symbols_evidencia = (
    tuple(symbol.upper() for symbol in SYMBOLS_OVERRIDE)
    if SYMBOLS_OVERRIDE
    else val._load_default_derivatives_symbols()
)

run_meta = {
    "fecha_hora_utc": RUN_TIMESTAMP.isoformat(),
    "raiz_proyecto": str(ROOT),
    "python": platform.python_version(),
    "plataforma": platform.platform(),
}
run_meta

{'fecha_hora_utc': '2026-06-03T23:03:45.853111+00:00',
 'raiz_proyecto': '/app',
 'python': '3.10.19',
 'plataforma': 'Linux-5.15.133.1-microsoft-standard-WSL2-x86_64-with-glibc2.36'}

## 2. Ejecución del validador end-to-end

En esta celda se invoca `scripts/validate_derivatives_pipeline.py` reutilizando exactamente la misma lógica que la *pipeline* de CI, pero adaptada al contexto interactivo:

- `importlib.reload(val)` recarga el módulo desde disco para que las **etiquetas** y la **matriz de aceptación** coincidan siempre con la versión actual del repositorio (evita falsos `FAIL` por caché del *kernel* entre ediciones).
- `stdout`/`stderr` se redirigen a un buffer en la propia celda para no saturar la salida con el logging técnico del validador (no se conserva en una variable persistente fuera de esta ejecución).
- Si la conexión a SQL falla, las suites aguas abajo se marcan como `SKIP` y la matriz informa qué precondiciones no se cumplieron.
- Si las tablas `derivatives_data.*` están listas, se ejecutan en cadena: suite `integration` -> suite `db_integration` -> `data_quality` (deduplicación + dominio/rango) -> `freeze coverage` (`check_target_coverage` resuelve la fecha con `DERIVATIVES_FREEZE_COVERAGE_DATE`; anulación solo vía `--target-date` en CLI) -> auditoría (`ingestion_runs`/`ingestion_events`) -> trazabilidad MLflow.

El resultado es la tabla `summary_df`, el semáforo de alto nivel del ETL: una fila por comprobación con su estado (`PASS`/`FAIL`/`SKIP`), si es requerida y un detalle legible.

In [2]:
technical_logs_buffer = io.StringIO()
with contextlib.redirect_stdout(technical_logs_buffer), contextlib.redirect_stderr(technical_logs_buffer):
    val.results.clear()

    db_ready = val.check_db_connection()
    val.check_derivatives_tables()
    val.run_integration_suite()

    if db_ready and val.results.get("TABLES_READY") == val.PASS:
        val.run_db_integration_suite()
        val.check_data_quality(symbols=symbols_evidencia)
        val.check_target_coverage(target_date=None, symbols=symbols_evidencia)
    else:
        val.results["DB_INTEGRATION"] = val.SKIP
        val.set_status(val.results, check_id="DATA_QUALITY_MIN", status=val.SKIP, aliases=("DATA_QUALITY",))
        val.results["FREEZE_COVERAGE"] = val.SKIP

    val.check_audit_evidence(db_integration_passed=val.results.get("DB_INTEGRATION") == val.PASS)
    val.check_mlflow_evidence()

    acceptance_matrix = val._build_acceptance_matrix(
        target_date=val.DERIVATIVES_FREEZE_COVERAGE_DATE,
        db_ready=db_ready,
        deriv_tables_ready=val.results.get("TABLES_READY") == val.PASS,
    )
    accepted, actionable_failures = val._print_summary(acceptance_matrix)

_ = technical_logs_buffer.getvalue()
print("Ejecución completada. Se ocultó el log técnico detallado.")

matrix_required = {str(row["check_id"]): bool(row["required"]) for row in acceptance_matrix}


def detalle_check(check_id: str, status: str) -> str:
    if status == val.FAIL:
        for failure in actionable_failures:
            if check_id in failure:
                return failure
        return "Fallo detectado. Revisar logs de validación para diagnóstico."

    if status == val.SKIP:
        return "No ejecutado por precondiciones no cumplidas."

    detalles_ok = {
        "INTEGRATION": "Suite integration ejecutada correctamente.",
        "DB_INTEGRATION": "Suite db_integration ejecutada correctamente.",
        "AUDIT_RUNS": "Se validó evidencia de ejecuciones en tabla de auditoría.",
        "AUDIT_EVENTS": "Se validó evidencia de eventos asociados a ejecuciones.",
        "MLFLOW_TRACKING": "Se encontró trazabilidad de ejecuciones del pipeline de derivados en MLflow.",
        "DATA_QUALITY_MIN": "Sin duplicados ni violaciones de dominio/rango (temporalidad al freeze: FREEZE_COVERAGE).",
        "FREEZE_COVERAGE": (
            f"Por tabla/símbolo, max(timestamp) >= bucket freeze 20:00 UTC del {DERIVATIVES_FREEZE_COVERAGE_DATE} "
            f"(FREEZE_COVERAGE / DERIVATIVES_FREEZE_COVERAGE_DATE)."
        ),
    }
    return detalles_ok.get(check_id, "Comprobación validada correctamente.")


summary_rows = []
for check_id, label in val._CHECK_LABELS.items():
    status = val.results.get(check_id, val.SKIP)
    summary_rows.append(
        {
            "id_check": check_id,
            "etiqueta": label,
            "estado": status,
            "requerido": "Sí" if matrix_required.get(check_id, False) else "No",
            "detalle": detalle_check(check_id, status),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

Ejecución completada. Se ocultó el log técnico detallado.


,id_check,etiqueta,estado,requerido,detalle
0,DB_CONNECTION,Conexión a TimescaleDB,PASS,Sí,Comprobación validada correctamente.
1,TABLES_READY,Tablas derivatives_data disponibles,PASS,Sí,Comprobación validada correctamente.
2,INTEGRATION,Suite de integración de derivados,PASS,Sí,Suite integration ejecutada correctamente.
3,DB_INTEGRATION,Suite de integración con BD de derivados,PASS,Sí,Suite db_integration ejecutada correctamente.
4,AUDIT_RUNS,Evidencia en derivatives_data.ingestion_runs,PASS,Sí,Se validó evidencia de ejecuciones en tabla de...
5,AUDIT_EVENTS,Evidencia en derivatives_data.ingestion_events,PASS,Sí,Se validó evidencia de eventos asociados a eje...
6,MLFLOW_TRACKING,Trazabilidad de runs de derivados en MLflow,PASS,No,Se encontró trazabilidad de ejecuciones del pi...
7,DATA_QUALITY_MIN,Deduplicación PK + dominio/rango,PASS,Sí,Sin duplicados ni violaciones de dominio/rango...
8,FREEZE_COVERAGE,Cobertura por tabla/símbolo hasta bucket objet...,PASS,Sí,"Por tabla/símbolo, max(timestamp) >= bucket fr..."


### 2.1. Lectura de la matriz de aceptación

Las comprobaciones publicadas por el validador cubren las cuatro capas críticas del ETL de derivados:

| Capa | Comprobaciones asociadas | Por qué importa |
|---|---|---|
| **Infraestructura** | `DB_CONNECTION`, `TABLES_READY` | Sin BD viva ni las tres fact tables creadas, no hay nada que validar |
| **Pipeline** | `INTEGRATION`, `DB_INTEGRATION` | Las suites de `integration` y `db_integration` corren de extremo a extremo: ingesta + persistencia con esquema y casing correctos |
| **Calidad de datos** | `DATA_QUALITY_MIN`, `FREEZE_COVERAGE` | Sin duplicados ni violaciones de dominio/rango; cobertura por tabla/símbolo **hasta el bucket freeze 4h** de **`DERIVATIVES_FREEZE_COVERAGE_DATE`** (2026-02-28 20:00 UTC) |
| **Auditoría / Trazabilidad** | `AUDIT_RUNS`, `AUDIT_EVENTS`, `MLFLOW_TRACKING` | Cada ejecución queda registrada en `ingestion_runs`/`ingestion_events` y reflejada en MLflow para reconstruir lo que se ingestó y cuándo |

En la corrida de evidencia (**2026-06-03**, `stamp` `20260603T230345Z`) las **nueve** comprobaciones figuran en `PASS`; `MLFLOW_TRACKING` es no requerido (acompaña al *backbone* de CI/CD pero no bloquea la aceptación). El export cierra con resultado global `PASS`. Si alguna fallara, el campo `detalle` apunta al motivo concreto: tabla vacía, símbolo huérfano, ejecución sin métrica MLflow, etc.


## 3. Evidencia SQL: cobertura, auditoría e integridad

Más allá del semáforo, conviene aterrizar la evidencia en **números concretos** sobre las tres tablas. Esta sección lanza **cuatro** consultas SQL:

1. **Cobertura por tabla y símbolo**: una `UNION ALL` sobre `fact_funding_rate`, `fact_open_interest_4h` y `fact_long_short_ratio_4h` con `MIN`/`MAX` del timestamp y conteo de filas. 
2. **Ejecuciones del cohorte de derivados en `ingestion_runs` (resumen por modo)**: consulta SQL **aparte**, agregación **global** sobre todas las filas con `pipeline_type` en (`etl_derivatives`, `etl_derivatives_live`, `etl_derivatives_backfill`).
3. **Duplicados por PK** por tabla: `(symbol, funding_time)` para funding y `(symbol, bucket_4h)` para OI y L/S. Debe ser estrictamente cero en las tres.
4. **Integridad real vs. imputaciones por tabla y símbolo**: una fila por **(fact table, símbolo)** con `pct_integridad_real` calculado sobre `is_imputed`. No se promedia entre fuentes con frecuencias distintas.

Adicionalmente se construye un **inventario en disco** de los `parquet` raw `<SIMBOLO>_<métrica>.parquet` bajo `data/01_raw/derivatives/` (uno por par símbolo×métrica), incluyendo `min`/`max` del timestamp leído del propio Parquet. Ese respaldo permite reconstruir las fact tables sin volver a llamar a la API REST de Binance Futures.

In [3]:
resumen_modos_df = pd.DataFrame()
duplicados_pk_df = pd.DataFrame()
integridad_df = pd.DataFrame()
parquet_inventory_df = pd.DataFrame()
vista_cobertura_filas_df = pd.DataFrame(
    columns=["símbolo", "tabla", "filas", "min_timestamp", "max_timestamp"]
)

_DERIV_PT_SQL = "'etl_derivatives', 'etl_derivatives_live', 'etl_derivatives_backfill'"

sql_logs_buffer = io.StringIO()
with contextlib.redirect_stdout(sql_logs_buffer), contextlib.redirect_stderr(sql_logs_buffer):
    engine = create_db_engine()
_ = sql_logs_buffer.getvalue()

if engine:
    cobertura_query = text(
        """
        WITH cobertura AS (
            SELECT 'fact_funding_rate' AS tabla, symbol,
                   MIN(funding_time) AS min_ts, MAX(funding_time) AS max_ts, COUNT(*) AS filas
            FROM derivatives_data.fact_funding_rate GROUP BY symbol
            UNION ALL
            SELECT 'fact_open_interest_4h', symbol,
                   MIN(bucket_4h), MAX(bucket_4h), COUNT(*)
            FROM derivatives_data.fact_open_interest_4h GROUP BY symbol
            UNION ALL
            SELECT 'fact_long_short_ratio_4h', symbol,
                   MIN(bucket_4h), MAX(bucket_4h), COUNT(*)
            FROM derivatives_data.fact_long_short_ratio_4h GROUP BY symbol
        )
        SELECT tabla, symbol, filas, min_ts, max_ts
        FROM cobertura ORDER BY tabla, symbol;
        """
    )
    cobertura_esperada = [(table_name, symbol) for table_name in DERIVATIVES_FACT_TABLES for symbol in symbols_evidencia]
    cobertura = pd.DataFrame(cobertura_esperada, columns=["tabla", "symbol"]).merge(
        pd.read_sql(cobertura_query, engine), on=["tabla", "symbol"], how="left",
    )
    cobertura["filas"] = cobertura["filas"].fillna(0).astype(int)

    vista_cobertura_filas_df = cobertura.assign(
        tabla=lambda dataframe: DERIVATIVES_SCHEMA + "." + dataframe["tabla"].astype(str),
        símbolo=lambda dataframe: dataframe["symbol"],
    ).rename(columns={"min_ts": "min_timestamp", "max_ts": "max_timestamp"})[
        ["símbolo", "tabla", "filas", "min_timestamp", "max_timestamp"]
    ]

    if not vista_cobertura_filas_df.empty:
        vista_cobertura_filas_df["min_timestamp"] = pd.to_datetime(
            vista_cobertura_filas_df["min_timestamp"], utc=True, errors="coerce"
        )
        vista_cobertura_filas_df["max_timestamp"] = pd.to_datetime(
            vista_cobertura_filas_df["max_timestamp"], utc=True, errors="coerce"
        )

    # Resumen agregado por modo sobre ingestion_runs del cohorte de derivados
    resumen_modos_query = text(
        f"""
        SELECT
            CASE
                WHEN LOWER(CAST(r.pipeline_type AS TEXT)) LIKE '%live%' THEN 'live'
                ELSE 'histórico'
            END AS modo,
            COUNT(*)::bigint AS runs,
            MAX(r.started_at) AS ultimo_inicio,
            COALESCE(SUM(r.total_rows), 0)::bigint AS total_filas,
            COALESCE(SUM(r.failed_count), 0)::bigint AS total_fallos_eventos,
            SUM(CASE WHEN COALESCE(r.failed_count, 0) > 0 THEN 1 ELSE 0 END)::bigint AS runs_con_fallo
        FROM derivatives_data.ingestion_runs r
        WHERE r.pipeline_type IN ({_DERIV_PT_SQL})
        GROUP BY 1
        ORDER BY modo;
        """
    )
    resumen_modos_df = pd.read_sql(resumen_modos_query, engine)
    if not resumen_modos_df.empty:
        resumen_modos_df["pct_runs_con_fallo"] = (
            (resumen_modos_df["runs_con_fallo"] / resumen_modos_df["runs"]) * 100.0
        ).round(2)

    # Verifica los duplicados por clave primaria en cada tabla
    duplicados_query = text(
        """
        SELECT 'fact_funding_rate' AS tabla, symbol,
               COUNT(*) AS total_filas,
               COUNT(DISTINCT funding_time) AS timestamps_unicos,
               COUNT(*) - COUNT(DISTINCT funding_time) AS duplicados_pk
        FROM derivatives_data.fact_funding_rate GROUP BY symbol
        UNION ALL
        SELECT 'fact_open_interest_4h', symbol,
               COUNT(*),
               COUNT(DISTINCT bucket_4h),
               COUNT(*) - COUNT(DISTINCT bucket_4h)
        FROM derivatives_data.fact_open_interest_4h GROUP BY symbol
        UNION ALL
        SELECT 'fact_long_short_ratio_4h', symbol,
               COUNT(*),
               COUNT(DISTINCT bucket_4h),
               COUNT(*) - COUNT(DISTINCT bucket_4h)
        FROM derivatives_data.fact_long_short_ratio_4h GROUP BY symbol
        ORDER BY 1, 2;
        """
    )
    duplicados_pk_df = pd.read_sql(duplicados_query, engine)

    # Calcula la integridad y las imputaciones por tabla de hechos y símbolo
    integridad_query = text(
        """
        SELECT 'fact_funding_rate' AS tabla, symbol,
               COUNT(*)                                             AS total_filas,
               SUM(CASE WHEN is_imputed THEN 1 ELSE 0 END)           AS filas_imputadas,
               SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END)      AS filas_reales,
               ROUND(
                   100.0 * SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END)
                   / NULLIF(COUNT(*), 0), 4
               )                                                     AS pct_integridad_real
        FROM derivatives_data.fact_funding_rate
        GROUP BY symbol
        UNION ALL
        SELECT 'fact_open_interest_4h', symbol,
               COUNT(*),
               SUM(CASE WHEN is_imputed THEN 1 ELSE 0 END),
               SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END),
               ROUND(
                   100.0 * SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END)
                   / NULLIF(COUNT(*), 0), 4
               )
        FROM derivatives_data.fact_open_interest_4h
        GROUP BY symbol
        UNION ALL
        SELECT 'fact_long_short_ratio_4h', symbol,
               COUNT(*),
               SUM(CASE WHEN is_imputed THEN 1 ELSE 0 END),
               SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END),
               ROUND(
                   100.0 * SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END)
                   / NULLIF(COUNT(*), 0), 4
               )
        FROM derivatives_data.fact_long_short_ratio_4h
        GROUP BY symbol
        ORDER BY tabla, symbol;
        """
    )
    integridad_df = pd.read_sql(integridad_query, engine).rename(columns={"symbol": "símbolo"})
    integridad_df["tabla"] = DERIVATIVES_SCHEMA + "." + integridad_df["tabla"].astype(str)
    integridad_df = integridad_df[
        [
            "tabla",
            "símbolo",
            "total_filas",
            "filas_imputadas",
            "filas_reales",
            "pct_integridad_real",
        ]
    ]

# Construye el inventario Parquet de artefactos raw de derivados
parquet_rows: list[dict] = []
for symbol in symbols_evidencia:
    for metric in DERIVATIVES_METRICS:
        parquet_path = DERIVATIVES_RAW_DIR / f"{symbol}_{metric}.parquet"
        if parquet_path.exists():
            parquet_size_mb = parquet_path.stat().st_size / (1024 * 1024)
            try:
                parquet_df = pd.read_parquet(parquet_path)
                timestamp_column = next(
                    (column for column in ("funding_time", "bucket_4h", "timestamp") if column in parquet_df.columns),
                    None,
                )
                parquet_timestamps = (
                    pd.to_datetime(parquet_df[timestamp_column], utc=True, errors="coerce").dropna()
                    if timestamp_column else pd.Series(dtype="datetime64[ns, UTC]")
                )
                parquet_rows.append(
                    {
                        "tipo": f"raw_{metric}",
                        "symbol": symbol,
                        "archivo": parquet_path.name,
                        "tamaño_mb": round(parquet_size_mb, 2),
                        "filas": len(parquet_df),
                        "columnas": len(parquet_df.columns),
                        "min_timestamp": parquet_timestamps.min() if not parquet_timestamps.empty else None,
                        "max_timestamp": parquet_timestamps.max() if not parquet_timestamps.empty else None,
                    }
                )
            except Exception:
                parquet_rows.append(
                    {
                        "tipo": f"raw_{metric}",
                        "symbol": symbol,
                        "archivo": parquet_path.name,
                        "tamaño_mb": round(parquet_size_mb, 2),
                        "filas": None,
                        "columnas": None,
                        "min_timestamp": None,
                        "max_timestamp": None,
                    }
                )
        else:
            parquet_rows.append(
                {
                    "tipo": f"raw_{metric}",
                    "symbol": symbol,
                    "archivo": f"{symbol}_{metric}.parquet",
                    "tamaño_mb": None,
                    "filas": None,
                    "columnas": None,
                    "min_timestamp": None,
                    "max_timestamp": None,
                }
            )

parquet_inventory_df = pd.DataFrame(parquet_rows)

## 4. Resultados y análisis

A continuación se imprimen en orden: **resumen agregado** de `ingestion_runs` por modo (histórico vs live), **cobertura** por tabla y símbolo, **duplicados** por PK, **integridad** e **inventario Parquet** en disco.

In [4]:
print("Resumen de ejecuciones derivados por modo (ingestion_runs, agregado global)")
display(resumen_modos_df)

print("\nCobertura por tabla y símbolo")
display(vista_cobertura_filas_df)

print("\nDuplicados por PK (por tabla)")
display(duplicados_pk_df)
if not duplicados_pk_df.empty and (duplicados_pk_df["duplicados_pk"] == 0).all():
    print("Cero duplicados confirmados para todas las tablas y símbolos")
else:
    print("Se detectaron duplicados o la consulta no devolvió resultados.")

print("\nIntegridad e imputaciones por tabla y símbolo (una fila por fact table)")
display(integridad_df)

print("\nInventario Parquet raw (derivados) en disco")
display(parquet_inventory_df)

Resumen de ejecuciones derivados por modo (ingestion_runs, agregado global)


,modo,runs,ultimo_inicio,total_filas,total_fallos_eventos,runs_con_fallo,pct_runs_con_fallo
0,histórico,24,2026-06-01 18:59:27.767070+00:00,95284,23,19,79.17
1,live,59,2026-05-31 03:22:10.690373+00:00,2826,30,3,5.08



Cobertura por tabla y símbolo


,símbolo,tabla,filas,min_timestamp,max_timestamp
0,BTCUSDT,derivatives_data.fact_funding_rate,7365,2019-09-10 08:00:00+00:00,2026-05-31 00:00:00+00:00
1,ETHUSDT,derivatives_data.fact_funding_rate,7131,2019-11-27 08:00:00+00:00,2026-05-31 00:00:00+00:00
2,BNBUSDT,derivatives_data.fact_funding_rate,6906,2020-02-10 08:00:00+00:00,2026-05-31 00:00:00+00:00
3,XRPUSDT,derivatives_data.fact_funding_rate,7011,2020-01-06 08:00:00+00:00,2026-05-31 00:00:00+00:00
4,SOLUSDT,derivatives_data.fact_funding_rate,6332,2020-09-13 16:00:00.004000+00:00,2026-05-31 00:00:00+00:00
5,BTCUSDT,derivatives_data.fact_open_interest_4h,9667,2022-01-01 00:00:00+00:00,2026-05-31 00:00:00+00:00
6,ETHUSDT,derivatives_data.fact_open_interest_4h,9673,2022-01-01 00:00:00+00:00,2026-06-01 00:00:00+00:00
7,BNBUSDT,derivatives_data.fact_open_interest_4h,9673,2022-01-01 00:00:00+00:00,2026-06-01 00:00:00+00:00
8,XRPUSDT,derivatives_data.fact_open_interest_4h,9673,2022-01-01 00:00:00+00:00,2026-06-01 00:00:00+00:00
9,SOLUSDT,derivatives_data.fact_open_interest_4h,9673,2022-01-01 00:00:00+00:00,2026-06-01 00:00:00+00:00



Duplicados por PK (por tabla)


,tabla,symbol,total_filas,timestamps_unicos,duplicados_pk
0,fact_funding_rate,BNBUSDT,6906,6906,0
1,fact_funding_rate,BTCUSDT,7365,7365,0
2,fact_funding_rate,ETHUSDT,7131,7131,0
3,fact_funding_rate,SOLUSDT,6332,6332,0
4,fact_funding_rate,XRPUSDT,7011,7011,0
5,fact_long_short_ratio_4h,BNBUSDT,9562,9562,0
6,fact_long_short_ratio_4h,BTCUSDT,9556,9556,0
7,fact_long_short_ratio_4h,ETHUSDT,9562,9562,0
8,fact_long_short_ratio_4h,SOLUSDT,9562,9562,0
9,fact_long_short_ratio_4h,XRPUSDT,9562,9562,0


Cero duplicados confirmados para todas las tablas y símbolos

Integridad e imputaciones por tabla y símbolo (una fila por fact table)


,tabla,símbolo,total_filas,filas_imputadas,filas_reales,pct_integridad_real
0,derivatives_data.fact_funding_rate,BNBUSDT,6906,0,6906,100.0000
1,derivatives_data.fact_funding_rate,BTCUSDT,7365,0,7365,100.0000
2,derivatives_data.fact_funding_rate,ETHUSDT,7131,0,7131,100.0000
3,derivatives_data.fact_funding_rate,SOLUSDT,6332,0,6332,100.0000
4,derivatives_data.fact_funding_rate,XRPUSDT,7011,0,7011,100.0000
5,derivatives_data.fact_long_short_ratio_4h,BNBUSDT,9562,0,9562,100.0000
6,derivatives_data.fact_long_short_ratio_4h,BTCUSDT,9556,156,9400,98.3675
7,derivatives_data.fact_long_short_ratio_4h,ETHUSDT,9562,0,9562,100.0000
8,derivatives_data.fact_long_short_ratio_4h,SOLUSDT,9562,0,9562,100.0000
9,derivatives_data.fact_long_short_ratio_4h,XRPUSDT,9562,0,9562,100.0000



Inventario Parquet raw (derivados) en disco


,tipo,symbol,archivo,tamaño_mb,filas,columnas,min_timestamp,max_timestamp
0,raw_funding_rate,BTCUSDT,BTCUSDT_funding_rate.parquet,0.13,7334,6,2019-09-10 08:00:00+00:00,2026-05-20 16:00:00+00:00
1,raw_open_interest_4h,BTCUSDT,BTCUSDT_open_interest_4h.parquet,0.27,9606,6,2022-01-01 00:00:00+00:00,2026-05-20 20:00:00+00:00
2,raw_long_short_ratio_4h,BTCUSDT,BTCUSDT_long_short_ratio_4h.parquet,0.36,9495,13,2022-01-19 12:00:00+00:00,2026-05-20 20:00:00+00:00
3,raw_funding_rate,ETHUSDT,ETHUSDT_funding_rate.parquet,0.13,7131,6,2019-11-27 08:00:00+00:00,2026-05-31 00:00:00+00:00
4,raw_open_interest_4h,ETHUSDT,ETHUSDT_open_interest_4h.parquet,0.27,9673,6,2022-01-01 00:00:00+00:00,2026-06-01 00:00:00+00:00
5,raw_long_short_ratio_4h,ETHUSDT,ETHUSDT_long_short_ratio_4h.parquet,0.36,9562,13,2022-01-19 12:00:00+00:00,2026-06-01 00:00:00+00:00
6,raw_funding_rate,BNBUSDT,BNBUSDT_funding_rate.parquet,0.12,6906,6,2020-02-10 08:00:00+00:00,2026-05-31 00:00:00+00:00
7,raw_open_interest_4h,BNBUSDT,BNBUSDT_open_interest_4h.parquet,0.26,9673,6,2022-01-01 00:00:00+00:00,2026-06-01 00:00:00+00:00
8,raw_long_short_ratio_4h,BNBUSDT,BNBUSDT_long_short_ratio_4h.parquet,0.36,9562,13,2022-01-19 12:00:00+00:00,2026-06-01 00:00:00+00:00
9,raw_funding_rate,XRPUSDT,XRPUSDT_funding_rate.parquet,0.13,7011,6,2020-01-06 08:00:00+00:00,2026-05-31 00:00:00+00:00


### 4.1. Interpretación de los resultados

**Resumen histórico/live.** En esta ejecución (`stamp` `20260603T230345Z`, `fecha_hora_utc` **2026-06-03 23:03 UTC**) el cohorte suma **83** corridas en `ingestion_runs`: **24** en modo histórico (**28.9 %**) y **59** en live (**71.1 %**). Histórico: último inicio **2026-06-01 18:59 UTC**, **95,284** filas acumuladas reportadas, **23** fallos de evento y **79.17 %** de corridas con al menos un fallo (**19** corridas). Live: último inicio **2026-05-31 03:22 UTC**, **2,826** filas reportadas, **30** fallos de evento y **5.08 %** de corridas con fallo (**3** corridas). El histórico concentra la mayor parte de los fallos de evento de la fase de endurecimiento del ETL; en live el ratio por corrida es mucho menor.

**Cobertura por tabla y símbolo.** Un cero en `filas` para algún par `(tabla, símbolo)` es señal de tabla/activo huérfano. Lo esperable es ver tres filas por símbolo (una por fact table). **`max_timestamp`** refleja el último dato persistido en BD (puede superar el bucket freeze si hay series vivas).

**Duplicados por PK.** El contrato es **cero estricto** en las tres tablas. Cualquier valor positivo invalida la PK natural y obliga a reingestar el símbolo/tabla afectado.

**Integridad e imputaciones por tabla.** La columna `pct_integridad_real` muestra cuánto de cada fact table proviene del exchange directamente (`is_imputed = false`). En esta corrida el funding está al **100 %** en los cinco símbolos; OI y L/S muestran **156** filas imputadas solo en **BTCUSDT** (~**98.4 %** de integridad real), mientras que el resto permanece al **100 %**. Lo importante es que el porcentaje sea **estable y trazable**.

**Inventario Parquet raw.** Confirma la existencia del respaldo en disco (`<SIMBOLO>_<métrica>.parquet`) con `tamaño_mb`, `filas` y rangos temporales coherentes con la cobertura SQL. Cualquier `tamaño_mb = NaN` indica un *raw* esperado pero ausente.

## 5. Exportación de artefactos para reproducibilidad

Para que las cifras anteriores sean **citables y comparables entre corridas**, todo se serializa en `reports/validation/derivatives/` con un `stamp` UTC en el nombre. Por cada ejecución se genera:

- Un **JSON maestro** (`derivatives_validation_<stamp>.json`) con metadatos, `fecha_objetivo_freeze_resuelta`, `target_bucket_utc`, símbolos, resultado global, fallos accionables, `checks`, `vista_cobertura`, `modos`, duplicados, integridad e inventario Parquet.
- Seis **CSV** (mismo `<stamp>`, p. ej. `20260603T230345Z` en esta corrida): `derivatives_validation_summary_<stamp>.csv`, `derivatives_validation_vista_cobertura_<stamp>.csv`, `derivatives_validation_modos_<stamp>.csv`, `derivatives_validation_duplicados_<stamp>.csv`, `derivatives_validation_integridad_<stamp>.csv` y `derivatives_validation_parquet_inventory_<stamp>.csv`.

Se reporta al final, en formato compacto:

- el **resultado global** del validador (`PASS`/`FAIL`),
- si en el **resumen por modo** sobre `ingestion_runs` aparecen tanto `histórico` como `live`,
- si se ha confirmado **cero duplicados** en las tres fact tables,
- si la **integridad por (fact table, símbolo)** está completa (3 fact tables × N símbolos),
- y las rutas de todos los artefactos exportados.

In [5]:
DERIVATIVES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_paths = {
    "json": DERIVATIVES_OUTPUT_DIR / f"derivatives_validation_{RUN_STAMP}.json",
    "summary": DERIVATIVES_OUTPUT_DIR / f"derivatives_validation_summary_{RUN_STAMP}.csv",
    "vista_cobertura": DERIVATIVES_OUTPUT_DIR / f"derivatives_validation_vista_cobertura_{RUN_STAMP}.csv",
    "modos": DERIVATIVES_OUTPUT_DIR / f"derivatives_validation_modos_{RUN_STAMP}.csv",
    "duplicados": DERIVATIVES_OUTPUT_DIR / f"derivatives_validation_duplicados_{RUN_STAMP}.csv",
    "integridad": DERIVATIVES_OUTPUT_DIR / f"derivatives_validation_integridad_{RUN_STAMP}.csv",
    "parquet_inventory": DERIVATIVES_OUTPUT_DIR / f"derivatives_validation_parquet_inventory_{RUN_STAMP}.csv",
}

historico_ok = bool(not resumen_modos_df.empty and (resumen_modos_df["modo"] == "histórico").any())
live_ok = bool(not resumen_modos_df.empty and (resumen_modos_df["modo"] == "live").any())
duplicados_ok = bool(not duplicados_pk_df.empty and (duplicados_pk_df["duplicados_pk"] == 0).all())
expected_integrity_rows = len(DERIVATIVES_FACT_TABLES) * len(symbols_evidencia)
integridad_ok = bool(not integridad_df.empty and len(integridad_df) == expected_integrity_rows)

payload = {
    "metadata": run_meta,
    "fecha_objetivo_freeze_resuelta": DERIVATIVES_FREEZE_COVERAGE_DATE,
    "target_bucket_utc": val._target_bucket_start(DERIVATIVES_FREEZE_COVERAGE_DATE).isoformat(),
    "symbols_evidencia": list(symbols_evidencia),
    "resultado_global": "PASS" if accepted else "FAIL",
    "fallos_accionables": actionable_failures,
    "checks": summary_df.to_dict(orient="records"),
    "vista_cobertura": vista_cobertura_filas_df.to_dict(orient="records") if not vista_cobertura_filas_df.empty else [],
    "modos": resumen_modos_df.to_dict(orient="records") if not resumen_modos_df.empty else [],
    "completitud_etl": {
        "histórico_detectado": historico_ok,
        "live_detectado": live_ok,
    },
    "duplicados_pk": {
        "verificado": duplicados_ok,
        "cero_duplicados_todas_tablas": duplicados_ok,
        "detalle": duplicados_pk_df.to_dict(orient="records") if not duplicados_pk_df.empty else [],
    },
    "integridad_imputaciones": {
        "verificado": integridad_ok,
        "detalle": integridad_df.to_dict(orient="records") if not integridad_df.empty else [],
    },
    "inventario_parquet": {
        "archivos_encontrados": int((parquet_inventory_df["tamaño_mb"].notna()).sum()) if not parquet_inventory_df.empty else 0,
        "archivos_esperados": len(parquet_inventory_df) if not parquet_inventory_df.empty else 0,
        "detalle": parquet_inventory_df.to_dict(orient="records") if not parquet_inventory_df.empty else [],
    },
}

export_paths["json"].write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
summary_df.to_csv(export_paths["summary"], index=False)
if not vista_cobertura_filas_df.empty:
    vista_cobertura_filas_df.to_csv(export_paths["vista_cobertura"], index=False)
if not resumen_modos_df.empty:
    resumen_modos_df.to_csv(export_paths["modos"], index=False)
if not duplicados_pk_df.empty:
    duplicados_pk_df.to_csv(export_paths["duplicados"], index=False)
if not integridad_df.empty:
    integridad_df.to_csv(export_paths["integridad"], index=False)
if not parquet_inventory_df.empty:
    parquet_inventory_df.to_csv(export_paths["parquet_inventory"], index=False)

print(f"Resultado global: {'PASS' if accepted else 'FAIL'}")
print(f"Completitud histórico: {'OK' if historico_ok else 'PENDIENTE'}")
print(f"Completitud live: {'OK' if live_ok else 'PENDIENTE'}")
print(f"Duplicados = 0 verificado: {'OK' if duplicados_ok else 'PENDIENTE'}")
print(f"Integridad por tabla y símbolo: {'OK' if integridad_ok else 'PENDIENTE'}")
print(f"Evidencia JSON: {export_paths['json']}")
print(f"Resumen de comprobaciones CSV: {export_paths['summary']}")
if not vista_cobertura_filas_df.empty:
    print(f"Vista cobertura (filas por hecho) CSV: {export_paths['vista_cobertura']}")
if not resumen_modos_df.empty:
    print(f"Resumen ejecuciones por modo (ingestion_runs) CSV: {export_paths['modos']}")
if not duplicados_pk_df.empty:
    print(f"Duplicados PK CSV: {export_paths['duplicados']}")
if not integridad_df.empty:
    print(f"Integridad/imputaciones CSV: {export_paths['integridad']}")
if not parquet_inventory_df.empty:
    print(f"Inventario Parquet CSV: {export_paths['parquet_inventory']}")

if engine:
    engine.dispose()


Resultado global: PASS
Completitud histórico: OK
Completitud live: OK
Duplicados = 0 verificado: OK
Integridad por tabla y símbolo: OK
Evidencia JSON: /app/reports/validation/derivatives/derivatives_validation_20260603T230345Z.json
Resumen de comprobaciones CSV: /app/reports/validation/derivatives/derivatives_validation_summary_20260603T230345Z.csv
Vista cobertura (filas por hecho) CSV: /app/reports/validation/derivatives/derivatives_validation_vista_cobertura_20260603T230345Z.csv
Resumen ejecuciones por modo (ingestion_runs) CSV: /app/reports/validation/derivatives/derivatives_validation_modos_20260603T230345Z.csv
Duplicados PK CSV: /app/reports/validation/derivatives/derivatives_validation_duplicados_20260603T230345Z.csv
Integridad/imputaciones CSV: /app/reports/validation/derivatives/derivatives_validation_integridad_20260603T230345Z.csv
Inventario Parquet CSV: /app/reports/validation/derivatives/derivatives_validation_parquet_inventory_20260603T230345Z.csv


**Nota.** Las cifras o estados obtenidos durante este notebook dependen del estado del proyecto en el momento de ejecutar el cuaderno; en otra ejecución pueden cambiar los valores y la lectura.